# TEM-1 Activity Screen — Plate Reader Analysis

**Experiment:** `exec_tem1_activity_screen_hack_world_22_20260725_230432`  
**Reader:** ELx808 | **Date:** 2026-07-26 01:57 AM  
**Protocol:** 15-min kinetic @ 490 & 405 nm, 30-s intervals, 37 °C  
**Substrate:** Nitrocefin (2× working solution, 50 µM final)  
**Enzyme:** Purified TEM-1, 0.1 ng/µL in 20 µL → 40 µL final reaction  

> ⚠️ **Compound identity correction:** The source plate PHD215176 well contents differed from the
> workflow plan. The positive control (Clavulanic Acid, H7) was correct. All 9 test compound slots
> were mismatched. Compound names below reflect the **actual** PHD215176 well contents.

## Assay Layout — CORRECTED (actual PHD215176 compounds)

| Slot | Wells | Intended (plan) | **Actual (corrected)** | Conc |
|---|---|---|---|---|
| +ctrl | F3, F7, G11 | Clavulanic Acid (T19860) | ✅ **Clavulanic Acid (T19860)** | 1 µM |
| Vehicle | B3, B7, C11 | DMSO vehicle | DMSO vehicle | — |
| −ctrl | D3, D7, E11 | BLB only | BLB only | — |
| 1 | B2, D2, F2 | Tazobactam (T1262) | **Cefpiramide acid (T0138)** | 50 µM |
| 2 | B4, D4, F4 | Sulbactam sodium (T6685) | **Ceftiofur sodium (T0198)** | 50 µM |
| 3 | B6, D6, F6 | Enmetazobactam (T14081) | **Cephradine (T0199)** | 50 µM |
| 4 | B8, D8, F8 | Amoxicillin (T1005) | **Methicillin sodium salt (T0234)** | 50 µM |
| 5 | B10, D10, F10 | Cephalexin (T1008) | **Cefadroxil (T0366)** | 50 µM |
| 6 | C5, E5, G5 | Meropenem (T0224) | **Dicloxacillin sodium hydrate (T1001)** | 50 µM |
| 7 | C9, E9, G9 | Oxacillin sodium (T0985) | **Amoxicillin (T1005)** | 50 µM |
| 8 | C3, E3, G3 | Cefpiramide acid (T0138) | **Cephalexin (T1008)** | 50 µM |
| 9 | C7, E7, G7 | Cefazolin (T8390) | **Cloxacillin sodium monohydrate (T1031)** | 50 µM |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path

DATA_DIR = Path('data')

# Load CSVs
df490 = pd.read_csv(DATA_DIR / '490nm_kinetic.csv', index_col='time_sec')
df405 = pd.read_csv(DATA_DIR / '405nm_kinetic.csv', index_col='time_sec')
plate_map = pd.read_csv(DATA_DIR / 'plate_map.csv', index_col='well')

# Drop the time_hms text column (keep numeric index)
df490 = df490.drop(columns=['time_hms'])
df405 = df405.drop(columns=['time_hms'])

print(f'490 nm: {df490.shape[0]} timepoints × {df490.shape[1]} wells')
print(f'405 nm: {df405.shape[0]} timepoints × {df405.shape[1]} wells')
print(f'Plate map: {len(plate_map)} wells annotated')
print('\nConditions:')
print(plate_map['condition'].value_counts().to_string())

## 1 · Plate Heatmap — Endpoint Absorbance (490 nm)

End-point absorbance across the whole plate at the final timepoint (t = 15 min).

In [ ]:
def make_plate_matrix(df_row, rows='ABCDEFGH', cols=range(1, 13)):
    """Reshape a single-row series (well -> value) into an 8×12 plate matrix."""
    mat = np.full((8, 12), np.nan)
    for well, val in df_row.items():
        if isinstance(well, str) and len(well) >= 2:
            r = rows.find(well[0])
            c = int(well[1:]) - 1
            if 0 <= r < 8 and 0 <= c < 12:
                mat[r, c] = val
    return mat

endpoint_490 = df490.iloc[-1]  # t=900 s
endpoint_405 = df405.iloc[-1]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, data, wl in zip(axes, [endpoint_490, endpoint_405], [490, 405]):
    mat = make_plate_matrix(data)
    im = ax.imshow(mat, cmap='RdYlGn_r' if wl == 490 else 'RdYlGn_r',
                   vmin=0.03, vmax=0.45, aspect='auto')
    ax.set_xticks(range(12))
    ax.set_xticklabels(range(1, 13))
    ax.set_yticks(range(8))
    ax.set_yticklabels(list('ABCDEFGH'))
    ax.set_title(f'Endpoint Absorbance — {wl} nm  (t = 15 min)', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Absorbance')
    # Annotate values
    for r in range(8):
        for c in range(12):
            v = mat[r, c]
            if not np.isnan(v):
                ax.text(c, r, f'{v:.3f}', ha='center', va='center',
                        fontsize=5.5, color='black')

plt.suptitle('TEM-1 Nitrocefin Activity Screen — Full Plate Endpoint', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'plate_heatmap_endpoint.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plate_heatmap_endpoint.png')

## 2 · Kinetic Traces — 490 nm (Nitrocefin Hydrolysis)

Absorbance at 490 nm increases as nitrocefin is hydrolysed by TEM-1 (yellow → red product).  
Inhibition = lower slope relative to Vehicle+TEM-1 control.

In [ ]:
# Colour palette per condition — CORRECTED actual compound identities
CONDITION_COLORS = {
    'Vehicle_TEM1_Control':                  '#1f77b4',   # blue
    'Negative_Control_NoEnzyme':             '#7f7f7f',   # grey
    'Positive_Control_ClavulanicAcid_1uM':   '#d62728',   # red
    'T0138_CefpiramideAcid_50uM':            '#ff7f0e',   # orange
    'T0198_CeftiofurSodium_50uM':            '#2ca02c',   # green
    'T0199_Cephradine_50uM':                 '#9467bd',   # purple
    'T0234_MethicillinSodium_50uM':          '#8c564b',   # brown
    'T0366_Cefadroxil_50uM':                 '#e377c2',   # pink
    'T1001_DicloxacillinSodium_50uM':        '#17becf',   # cyan
    'T1005_Amoxicillin_50uM':                '#bcbd22',   # yellow-green
    'T1008_Cephalexin_50uM':                 '#aec7e8',   # light blue
    'T1031_CloxacillinSodium_50uM':          '#ffbb78',   # peach
    'Unassigned':                            '#cccccc',
}

CONDITION_LABELS = {
    'Vehicle_TEM1_Control':                  'Vehicle + TEM-1',
    'Negative_Control_NoEnzyme':             'No Enzyme',
    'Positive_Control_ClavulanicAcid_1uM':   'Clavulanic Acid 1 µM (+ctrl)',
    'T0138_CefpiramideAcid_50uM':            'Cefpiramide Acid 50 µM (T0138)',
    'T0198_CeftiofurSodium_50uM':            'Ceftiofur Sodium 50 µM (T0198)',
    'T0199_Cephradine_50uM':                 'Cephradine 50 µM (T0199)',
    'T0234_MethicillinSodium_50uM':          'Methicillin Na 50 µM (T0234)',
    'T0366_Cefadroxil_50uM':                 'Cefadroxil 50 µM (T0366)',
    'T1001_DicloxacillinSodium_50uM':        'Dicloxacillin Na·H₂O 50 µM (T1001)',
    'T1005_Amoxicillin_50uM':                'Amoxicillin 50 µM (T1005)',
    'T1008_Cephalexin_50uM':                 'Cephalexin 50 µM (T1008)',
    'T1031_CloxacillinSodium_50uM':          'Cloxacillin Na·H₂O 50 µM (T1031)',
    'Unassigned':                            'Unassigned',
}

times_min = np.array(df490.index) / 60.0  # convert seconds to minutes

fig, ax = plt.subplots(figsize=(14, 7))

plotted = set()
for well in df490.columns:
    cond = plate_map.loc[well, 'condition'] if well in plate_map.index else 'Unassigned'
    color = CONDITION_COLORS.get(cond, '#cccccc')
    label = CONDITION_LABELS.get(cond, cond) if cond not in plotted else '_'
    if cond not in plotted:
        plotted.add(cond)
    ax.plot(times_min, df490[well].values, color=color, alpha=0.7,
            linewidth=1.5, label=label)

ax.set_xlabel('Time (minutes)', fontsize=12)
ax.set_ylabel('Absorbance at 490 nm', fontsize=12)
ax.set_title('TEM-1 Nitrocefin Screen — Kinetic Traces (490 nm) [corrected compounds]',
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9, framealpha=0.8)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 15)
plt.tight_layout()
plt.savefig(DATA_DIR / 'kinetic_490nm_all_wells.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: kinetic_490nm_all_wells.png')

## 3 · Mean ± SEM Kinetic Traces per Condition

In [ ]:
def compute_mean_sem(df, plate_map, condition):
    wells = plate_map[plate_map['condition'] == condition].index.tolist()
    wells = [w for w in wells if w in df.columns]
    if not wells:
        return None, None, None
    sub = df[wells]
    return sub.mean(axis=1).values, sub.sem(axis=1).values, wells

fig, ax = plt.subplots(figsize=(14, 7))

# Priority display order — corrected compound keys
priority = [
    'Negative_Control_NoEnzyme',
    'Vehicle_TEM1_Control',
    'Positive_Control_ClavulanicAcid_1uM',
    'T0138_CefpiramideAcid_50uM',
    'T0198_CeftiofurSodium_50uM',
    'T0199_Cephradine_50uM',
    'T0234_MethicillinSodium_50uM',
    'T0366_Cefadroxil_50uM',
    'T1001_DicloxacillinSodium_50uM',
    'T1005_Amoxicillin_50uM',
    'T1008_Cephalexin_50uM',
    'T1031_CloxacillinSodium_50uM',
]

for cond in priority:
    mean_vals, sem_vals, wells = compute_mean_sem(df490, plate_map, cond)
    if mean_vals is None:
        continue
    color = CONDITION_COLORS.get(cond, '#cccccc')
    label = f"{CONDITION_LABELS[cond]} (n={len(wells)})"
    ax.plot(times_min, mean_vals, color=color, linewidth=2.5, label=label)
    ax.fill_between(times_min, mean_vals - sem_vals, mean_vals + sem_vals,
                    color=color, alpha=0.15)

ax.set_xlabel('Time (minutes)', fontsize=12)
ax.set_ylabel('Mean Absorbance at 490 nm', fontsize=12)
ax.set_title('TEM-1 Nitrocefin Screen — Mean ± SEM Kinetics (490 nm) [corrected compounds]',
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 15)
plt.tight_layout()
plt.savefig(DATA_DIR / 'kinetic_490nm_mean_sem.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: kinetic_490nm_mean_sem.png')

## 4 · Dual-Wavelength Kinetics (490 & 405 nm)

490 nm: nitrocefin hydrolysis product (red).  
405 nm: alternative tracking wavelength.

In [ ]:
times_min_405 = np.array(df405.index) / 60.0

# Key conditions for dual-wavelength plot — corrected
key_conditions = {
    'Vehicle_TEM1_Control':                'Vehicle + TEM-1',
    'Negative_Control_NoEnzyme':           'No Enzyme',
    'Positive_Control_ClavulanicAcid_1uM': 'Clavulanic Acid (+ctrl)',
    'T0138_CefpiramideAcid_50uM':          'Cefpiramide Acid 50 µM (T0138)',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, df, times, wl in zip(axes, [df490, df405], [times_min, times_min_405], [490, 405]):
    for cond, label in key_conditions.items():
        mean_vals, sem_vals, wells = compute_mean_sem(df, plate_map, cond)
        if mean_vals is None:
            continue
        color = CONDITION_COLORS.get(cond, '#cccccc')
        ax.plot(times, mean_vals, color=color, linewidth=2.5, label=f'{label} (n={len(wells)})')
        ax.fill_between(times, mean_vals - sem_vals, mean_vals + sem_vals,
                        color=color, alpha=0.2)
    ax.set_xlabel('Time (minutes)', fontsize=11)
    ax.set_ylabel(f'Absorbance at {wl} nm', fontsize=11)
    ax.set_title(f'{wl} nm — Key Conditions [corrected]', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 15)

plt.suptitle('TEM-1 Screen — Dual-Wavelength Kinetics (Key Conditions)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'dual_wavelength_kinetics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 · % Inhibition Calculation

Using endpoint absorbance at 490 nm:  
```
% Inhibition = 100 × (A_vehicle - A_compound) / (A_vehicle - A_no_enzyme)
```

In [ ]:
# Use mean of last 3 timepoints as endpoint (more robust than single point)
endpoint = df490.iloc[-3:].mean()

def get_mean_endpoint(condition):
    wells = plate_map[plate_map['condition'] == condition].index.tolist()
    wells = [w for w in wells if w in endpoint.index]
    if not wells:
        return np.nan, np.nan, []
    vals = endpoint[wells].values
    return np.mean(vals), np.std(vals, ddof=1), wells

vehicle_mean, vehicle_sd, vehicle_wells = get_mean_endpoint('Vehicle_TEM1_Control')
neg_mean, neg_sd, neg_wells = get_mean_endpoint('Negative_Control_NoEnzyme')
pos_mean, pos_sd, pos_wells = get_mean_endpoint('Positive_Control_ClavulanicAcid_1uM')

print(f"Vehicle + TEM-1:  {vehicle_mean:.4f} ± {vehicle_sd:.4f}  (wells: {vehicle_wells})")
print(f"No-Enzyme:        {neg_mean:.4f} ± {neg_sd:.4f}  (wells: {neg_wells})")
print(f"Pos Ctrl (Clav):  {pos_mean:.4f} ± {pos_sd:.4f}  (wells: {pos_wells})")

dynamic_range = vehicle_mean - neg_mean
print(f"\nDynamic range (Vehicle - NoEnzyme): {dynamic_range:.4f}")
print(f"Z' factor estimate (Pos ctrl): {1 - 3*(pos_sd + vehicle_sd)/abs(pos_mean - vehicle_mean):.3f}")

# Compute % inhibition per condition — CORRECTED compound keys
test_conditions = [
    'T0138_CefpiramideAcid_50uM',
    'T0198_CeftiofurSodium_50uM',
    'T0199_Cephradine_50uM',
    'T0234_MethicillinSodium_50uM',
    'T0366_Cefadroxil_50uM',
    'T1001_DicloxacillinSodium_50uM',
    'T1005_Amoxicillin_50uM',
    'T1008_Cephalexin_50uM',
    'T1031_CloxacillinSodium_50uM',
]

results = []
for cond in test_conditions:
    mean_ep, sd_ep, wells = get_mean_endpoint(cond)
    pct_inh = 100.0 * (vehicle_mean - mean_ep) / dynamic_range
    pct_sd = 100.0 * np.sqrt(sd_ep**2 + vehicle_sd**2) / dynamic_range
    results.append({
        'condition': cond,
        'label': CONDITION_LABELS.get(cond, cond),
        'n_replicates': len(wells),
        'mean_abs490': round(mean_ep, 4),
        'sd_abs490': round(sd_ep, 4),
        'pct_inhibition': round(pct_inh, 1),
        'pct_inh_sd': round(pct_sd, 1),
    })

# Add controls
results.insert(0, {
    'condition': 'Vehicle_TEM1_Control',
    'label': 'Vehicle + TEM-1 (0% ctrl)',
    'n_replicates': len(vehicle_wells),
    'mean_abs490': round(vehicle_mean, 4),
    'sd_abs490': round(vehicle_sd, 4),
    'pct_inhibition': 0.0,
    'pct_inh_sd': 0.0,
})
results.insert(1, {
    'condition': 'Positive_Control_ClavulanicAcid_1uM',
    'label': 'Clavulanic Acid 1µM (+ctrl)',
    'n_replicates': len(pos_wells),
    'mean_abs490': round(pos_mean, 4),
    'sd_abs490': round(pos_sd, 4),
    'pct_inhibition': round(100.0 * (vehicle_mean - pos_mean) / dynamic_range, 1),
    'pct_inh_sd': 0.0,
})

results_df = pd.DataFrame(results)
results_df.to_csv(DATA_DIR / 'inhibition_summary.csv', index=False)
print('\n── % Inhibition Summary (corrected compounds) ──')
print(results_df[['label', 'n_replicates', 'mean_abs490', 'sd_abs490',
                   'pct_inhibition', 'pct_inh_sd']].to_string(index=False))
print('\nSaved: inhibition_summary.csv')

## 6 · % Inhibition Bar Chart

In [ ]:
plot_df = results_df.sort_values('pct_inhibition', ascending=False).copy()

colors = [CONDITION_COLORS.get(c, '#cccccc') for c in plot_df['condition']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(plot_df)), plot_df['pct_inhibition'],
              yerr=plot_df['pct_inh_sd'],
              color=colors, edgecolor='black', linewidth=0.7,
              capsize=5, error_kw={'elinewidth': 1.5})

# Annotate values
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.text(i, row['pct_inhibition'] + row['pct_inh_sd'] + 1.5,
            f"{row['pct_inhibition']:.1f}%",
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(100, color='red', linewidth=0.8, linestyle='--', alpha=0.5, label='100% inhibition')
ax.axhline(50, color='orange', linewidth=0.8, linestyle=':', alpha=0.7, label='50% threshold')

ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df['label'], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('% Inhibition of TEM-1 Activity', fontsize=12)
ax.set_title('TEM-1 Nitrocefin Screen — % Inhibition at Endpoint (490 nm)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(min(plot_df['pct_inhibition'].min() - 15, -20), 130)

plt.tight_layout()
plt.savefig(DATA_DIR / 'inhibition_barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: inhibition_barchart.png')

## 7 · Linear Slopes (Initial Velocity)

Fit a linear slope to the first 5 minutes of the kinetic trace (pre-plateau linear regime) to estimate initial velocity (∆A490/min). Lower slope = more inhibition.

In [ ]:
# Use first 5 minutes (10 timepoints at 30s intervals)
n_early = min(10, len(times_min))
t_early = times_min[:n_early]

def fit_slope(df, plate_map, condition):
    wells = plate_map[plate_map['condition'] == condition].index.tolist()
    wells = [w for w in wells if w in df.columns]
    if not wells:
        return np.nan, np.nan
    slopes = []
    for w in wells:
        y = df[w].values[:n_early]
        slope, intercept, r, p, se = stats.linregress(t_early, y)
        slopes.append(slope)
    return np.mean(slopes), np.std(slopes, ddof=1)

# All conditions — corrected keys
all_conditions = ['Vehicle_TEM1_Control', 'Negative_Control_NoEnzyme',
                  'Positive_Control_ClavulanicAcid_1uM'] + test_conditions

slope_results = []
for cond in all_conditions:
    s_mean, s_sd = fit_slope(df490, plate_map, cond)
    slope_results.append({
        'condition': cond,
        'label': CONDITION_LABELS.get(cond, cond),
        'slope_mean': round(s_mean * 1000, 4),  # mAbs/min
        'slope_sd': round(s_sd * 1000, 4),
    })

slopes_df = pd.DataFrame(slope_results).sort_values('slope_mean', ascending=False)
slopes_df.to_csv(DATA_DIR / 'initial_velocity.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors2 = [CONDITION_COLORS.get(r['condition'], '#cccccc') for _, r in slopes_df.iterrows()]
ax.bar(range(len(slopes_df)), slopes_df['slope_mean'],
       yerr=slopes_df['slope_sd'],
       color=colors2, edgecolor='black', linewidth=0.7,
       capsize=5, error_kw={'elinewidth': 1.5})
ax.set_xticks(range(len(slopes_df)))
ax.set_xticklabels(slopes_df['label'], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Initial Velocity (mΔA490 / min, 0–5 min)', fontsize=11)
ax.set_title('TEM-1 Screen — Initial Velocity per Condition [corrected compounds]',
             fontsize=13, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / 'initial_velocity.png', dpi=150, bbox_inches='tight')
plt.show()
print(slopes_df[['label', 'slope_mean', 'slope_sd']].to_string(index=False))

## 8 · Per-Compound Replicate Kinetics (Faceted)

In [ ]:
# Faceted per-condition kinetics — corrected compound keys
all_conds_for_facet = [
    'Vehicle_TEM1_Control',
    'Positive_Control_ClavulanicAcid_1uM',
    'T0138_CefpiramideAcid_50uM',
    'T0198_CeftiofurSodium_50uM',
    'T0199_Cephradine_50uM',
    'T0234_MethicillinSodium_50uM',
    'T0366_Cefadroxil_50uM',
    'T1001_DicloxacillinSodium_50uM',
    'T1005_Amoxicillin_50uM',
    'T1008_Cephalexin_50uM',
    'T1031_CloxacillinSodium_50uM',
    'Negative_Control_NoEnzyme',
]

fig, axes = plt.subplots(3, 4, figsize=(18, 12), sharex=True, sharey=True)
axes = axes.flatten()

# Get vehicle mean for reference
veh_wells = plate_map[plate_map['condition'] == 'Vehicle_TEM1_Control'].index.tolist()
veh_wells = [w for w in veh_wells if w in df490.columns]
veh_mean_trace = df490[veh_wells].mean(axis=1).values

for ax, cond in zip(axes, all_conds_for_facet):
    wells = plate_map[plate_map['condition'] == cond].index.tolist()
    wells = [w for w in wells if w in df490.columns]
    color = CONDITION_COLORS.get(cond, '#cccccc')
    ax.plot(times_min, veh_mean_trace, color='#1f77b4', linewidth=1.0,
            linestyle='--', alpha=0.5, label='Vehicle ref')
    for w in wells:
        ax.plot(times_min, df490[w].values, color=color, linewidth=1.5,
                alpha=0.8, label=w)
    ax.set_title(CONDITION_LABELS.get(cond, cond), fontsize=8, fontweight='bold')
    ax.legend(fontsize=6, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 15)

for ax in axes[len(all_conds_for_facet):]:
    ax.set_visible(False)

fig.text(0.5, 0.02, 'Time (minutes)', ha='center', fontsize=12)
fig.text(0.01, 0.5, 'Absorbance 490 nm', va='center', rotation='vertical', fontsize=12)
plt.suptitle('TEM-1 Screen — Per-Condition Replicate Kinetics (490 nm) [corrected compounds]',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / 'faceted_kinetics_490nm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: faceted_kinetics_490nm.png')

## 9 · Summary Table

In [ ]:
summary = results_df.merge(
    slopes_df[['condition', 'slope_mean', 'slope_sd']],
    on='condition', how='left'
).sort_values('pct_inhibition', ascending=False)

summary_display = summary[['label', 'n_replicates', 'mean_abs490', 'sd_abs490',
                            'pct_inhibition', 'pct_inh_sd',
                            'slope_mean', 'slope_sd']].copy()
summary_display.columns = ['Condition', 'N', 'Mean A490', 'SD A490',
                            '% Inhibition', 'SD Inh',
                            'Slope (mA/min)', 'SD Slope']

summary_display.to_csv(DATA_DIR / 'full_summary.csv', index=False)
print(summary_display.to_string(index=False))
print('\nSaved: full_summary.csv')